<a href="https://colab.research.google.com/github/justii543/MLpreps/blob/main/Milestone_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import zipfile
import io

# Prompt to choose the zip file from your local machine
uploaded = files.upload()

# Extract the folder contents
for file_name in uploaded.keys():
    with zipfile.ZipFile(io.BytesIO(uploaded[file_name]), 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print(f"Successfully extracted: {file_name}")

With Unsafe API (80-20 Safe-Vulnerable samples)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    matthews_corrcoef
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json
import random
import re
import warnings
warnings.filterwarnings('ignore')

# 1. Unsafe api functions

unsafe_functions = [
    "strcpy", "strcat", "sprintf", "vsprintf", "gets", "scanf", "fscanf", "sscanf",
    "memcpy", "memmove", "bcopy", "wcscpy", "wcscat", "swprintf",
    "printf", "fprintf", "snprintf", "syslog", "vprintf", "vfprintf",
    "malloc", "calloc", "realloc", "free", "alloca", "mmap", "munmap",
    "system", "popen", "exec", "execl", "execle", "execlp", "execv", "execve", "execvp",
    "fopen", "open", "creat", "access", "stat", "lstat", "readlink", "unlink",
    "atoi", "atol", "strtol", "strtoul", "atof", "strtod",
    "setuid", "setgid", "chmod", "chown", "seteuid", "setegid",
]

def add_unsafe_api_tags(code):
    code_lower = code.lower()
    found = []
    for func in unsafe_functions:
        if re.search(r"\b" + re.escape(func) + r"\s*\(", code_lower):
            found.append(func)
    if found:
        code = code + "\n\n<unsafe_apis> " + " ".join(found) + " </unsafe_apis>"
    else:
        code = code + "\n\n<unsafe_apis> none </unsafe_apis>"
    return code

# 2. Load data
print("\n Load dataset")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

with open("/content/PrimeVul-main/primevul_train .jsonl", "r") as f:
    train_full = [json.loads(line) for line in f]

with open("/content/PrimeVul-main/primevul_test .jsonl", "r") as f:
    test_full = [json.loads(line) for line in f]

# 3. Training : Balanced Dataset
train_safe_all = [x for x in train_full if x['target'] == 0]
train_vuln_all = [x for x in train_full if x['target'] == 1]

random.shuffle(train_safe_all)
random.shuffle(train_vuln_all)

VULN_COUNT = min(len(train_vuln_all), 2000)
SAFE_COUNT = VULN_COUNT

train_data = train_safe_all[:SAFE_COUNT] + train_vuln_all[:VULN_COUNT]
random.shuffle(train_data)

print(f"Training: {SAFE_COUNT} Safe + {VULN_COUNT} Vuln = {len(train_data)}")

# 4. Test : 800 safe + 200 vulnerable
test_safe_all = [x for x in test_full if x['target'] == 0]
test_vuln_all = [x for x in test_full if x['target'] == 1]

random.shuffle(test_safe_all)
random.shuffle(test_vuln_all)

TEST_SAFE = 800
TEST_VULN = min(len(test_vuln_all), 200)

test_data = test_safe_all[:TEST_SAFE] + test_vuln_all[:TEST_VULN]
random.shuffle(test_data)

test_safe_count = TEST_SAFE
test_vuln_count = TEST_VULN

print(f"Test: {test_safe_count} Safe + {test_vuln_count} Vuln = {len(test_data)}")

# 5. Enhance data
print("\n Enhancing data with unsafe API tags...")
for item in train_data:
    item['func'] = add_unsafe_api_tags(item['func'])
for item in test_data:
    item['func'] = add_unsafe_api_tags(item['func'])

# 6. Tokenize
print("\n Tokenizing...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

train_codes = [x['func'] for x in train_data]
train_labels = [x['target'] for x in train_data]
test_codes = [item['func'] for item in test_data]
test_labels = [item['target'] for item in test_data]

train_encodings = tokenizer(train_codes, truncation=True, padding=True, max_length=256, return_tensors='pt')
test_encodings = tokenizer(test_codes, truncation=True, padding=True, max_length=256, return_tensors='pt')
print(f"Train: {train_encodings['input_ids'].shape}, Test: {test_encodings['input_ids'].shape}")

# 7. Dataset
from datasets import Dataset as HFDataset
train_dataset = HFDataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels
})

# 8. Focal loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


# 9. Model
print("\n Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/codebert-base", num_labels=2, ignore_mismatched_sizes=True
)
model.to(device)

focal_loss_fn = FocalLoss(alpha=0.25, gamma=2.0)

class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = focal_loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# 10. Train
print("\n Training with Focal Loss...")

training_args = TrainingArguments(
    output_dir="./focal_model_final",
    num_train_epochs=15,
    per_device_train_batch_size=16,
    learning_rate=5e-5,
    weight_decay=0.02,
    warmup_ratio=0.1,
    logging_steps=20,
    save_strategy="no",
    report_to="none",
    fp16=True if device.type == "cuda" else False,
    gradient_accumulation_steps=2,
    lr_scheduler_type="cosine_with_restarts",
    dataloader_pin_memory=False,
)

trainer = FocalLossTrainer(model=model, args=training_args, train_dataset=train_dataset)
trainer.train()
model.eval()


# 11. Predictions
print("\n Generating predictions")

test_dataset = TensorDataset(
    test_encodings['input_ids'],
    test_encodings['attention_mask'],
    torch.tensor(test_labels, dtype=torch.long)
)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

all_probabilities = []
all_labels_list = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=1)
        all_probabilities.extend(probs[:, 1].cpu().numpy())
        all_labels_list.extend(labels.numpy())

all_probabilities = np.array(all_probabilities)
all_labels = np.array(all_labels_list)

# 12. Optimal threshold
print("\n Finding optimal threshold")

best_f1 = 0
best_threshold = 0.5
best_predictions = None
best_precision = 0
best_recall = 0

for threshold in np.arange(0.01, 0.99, 0.005):
    predictions = (all_probabilities >= threshold).astype(int)
    if predictions.sum() < 5:
        continue

    prec = precision_score(all_labels, predictions, zero_division=0)
    rec = recall_score(all_labels, predictions, zero_division=0)
    f1_val = f1_score(all_labels, predictions, zero_division=0)

    if f1_val > best_f1:
        best_f1 = f1_val
        best_threshold = threshold
        best_predictions = predictions.copy()
        best_precision = prec
        best_recall = rec

print(f"Optimal threshold: {best_threshold:.3f}")
print(f"Best F1-Score: {best_f1:.4f}")

# 13. final metrics
all_predictions = best_predictions

tn, fp, fn, tp = confusion_matrix(all_labels, all_predictions).ravel()

accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions, zero_division=0)
recall = recall_score(all_labels, all_predictions, zero_division=0)
f1 = f1_score(all_labels, all_predictions, zero_division=0)
auc_roc = roc_auc_score(all_labels, all_probabilities)
avg_precision = average_precision_score(all_labels, all_probabilities)
mcc = matthews_corrcoef(all_labels, all_predictions)

precision_class, recall_class, f1_class, _ = precision_recall_fscore_support(
    all_labels, all_predictions, zero_division=0
)

# 14. results
print("Final Results")

print(f"""
╔══════════════════════════════════════╦════════════╗
║ METRIC                               ║ VALUE      ║
╠══════════════════════════════════════╬════════════╣
║ 1. Accuracy                          ║ {accuracy:.4f}    ║
║ 2. Precision (Vulnerable)            ║ {precision:.4f}    ║
║ 3. Recall / Sensitivity              ║ {recall:.4f}    ║
║ 4. F1-Score (Vulnerable)             ║ {f1:.4f}    ║
║ 5. AUC-ROC                           ║ {auc_roc:.4f}    ║
║ 6. MCC (Matthews Corr. Coeff.)       ║ {mcc:.4f}    ║
╚══════════════════════════════════════╩════════════╝
""")

print(f"""
 CONFUSION MATRIX:
┌────────────────────┬──────────────┬──────────────────┐
│                    │ Pred. SAFE   │ Pred. VULNERABLE │
├────────────────────┼──────────────┼──────────────────┤
│ Actually SAFE      │     {tn:<8} │       {fp:<10} │
│ Actually VULNERABLE│     {fn:<8} │       {tp:<10} │
└────────────────────┴──────────────┴──────────────────┘

Vulnerabilities Found: {tp}/{test_vuln_count} ({(tp/test_vuln_count*100) if test_vuln_count > 0 else 0:.1f}%)
Safe Code Correct: {tn}/{test_safe_count} ({(tn/test_safe_count*100) if test_safe_count > 0 else 0:.1f}%)
False Alarms: {fp}
 Missed Vulnerabilities: {fn}
""")

print(f"""
 CLASSIFICATION REPORT:
{classification_report(all_labels, all_predictions, target_names=['Safe (0)', 'Vulnerable (1)'], zero_division=0)}
""")

# 15. Visualization
print(" Generating visualizations...")

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
fig.suptitle(f'Vulnerability Detection Results (Focal Loss + Unsafe API Tags)\nAcc={accuracy:.4f} | Prec={precision:.4f} | Rec={recall:.4f} | F1={f1:.4f} | AUC={auc_roc:.4f} | MCC={mcc:.4f}',
             fontsize=11, fontweight='bold', y=0.98)

# 1. Confusion Matrix
sns.heatmap([[tn, fp], [fn, tp]], annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred SAFE', 'Pred VULNERABLE'],
            yticklabels=['True SAFE', 'True VULNERABLE'],
            ax=axes[0, 0], cbar_kws={'label': 'Count'},
            annot_kws={'size': 16, 'fontweight': 'bold'})
axes[0, 0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')

# 2. ROC Curve
fpr_curve, tpr_curve, _ = roc_curve(all_labels, all_probabilities)
axes[0, 1].plot(fpr_curve, tpr_curve, 'b-', linewidth=3, label=f'AUC = {auc_roc:.4f}')
axes[0, 1].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random')
axes[0, 1].fill_between(fpr_curve, tpr_curve, alpha=0.3, color='blue')
axes[0, 1].set_xlabel('False Positive Rate', fontsize=12)
axes[0, 1].set_ylabel('True Positive Rate', fontsize=12)
axes[0, 1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[0, 1].legend(loc='lower right', fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

# 3. Precision-Recall Curve
prec_curve, rec_curve, _ = precision_recall_curve(all_labels, all_probabilities)
axes[0, 2].plot(rec_curve, prec_curve, 'g-', linewidth=3, label=f'AP = {avg_precision:.4f}')
axes[0, 2].set_xlabel('Recall', fontsize=12)
axes[0, 2].set_ylabel('Precision', fontsize=12)
axes[0, 2].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[0, 2].legend(loc='upper right', fontsize=11)
axes[0, 2].grid(True, alpha=0.3)

# 4. 6 Key Metrics
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'MCC']
metric_vals = [accuracy, precision, recall, f1, auc_roc, mcc]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#1abc9c', '#9b59b6']
bars = axes[1, 0].bar(metric_names, metric_vals, color=colors, edgecolor='black', linewidth=2)
axes[1, 0].set_ylim([0, 1.15])
axes[1, 0].set_title('6 Key Performance Metrics', fontsize=14, fontweight='bold')
for bar, val in zip(bars, metric_vals):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
                    f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 5. Per-Class Performance
class_names = ['Safe (0)', 'Vulnerable (1)']
f1_vals = [f1_class[0], f1_class[1]]
prec_vals = [precision_class[0], precision_class[1]]
rec_vals = [recall_class[0], recall_class[1]]

x = np.arange(len(class_names))
width = 0.25
axes[1, 1].bar(x - width, prec_vals, width, label='Precision', color='#3498db', edgecolor='black')
axes[1, 1].bar(x, rec_vals, width, label='Recall', color='#e74c3c', edgecolor='black')
axes[1, 1].bar(x + width, f1_vals, width, label='F1-Score', color='#2ecc71', edgecolor='black')
axes[1, 1].set_xlabel('Class', fontsize=12)
axes[1, 1].set_title('Per-Class Performance', fontsize=14, fontweight='bold')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(class_names, fontsize=11)
axes[1, 1].legend(fontsize=11)
axes[1, 1].set_ylim([0, 1.1])
axes[1, 1].grid(True, alpha=0.3, axis='y')

# 6. Pie Chart
error_labels = ['True Positive\n(Vuln Detected)', 'True Negative\n(Safe Detected)',
                'False Negative\n(Vuln Missed)', 'False Positive\n(False Alarm)']
error_vals = [tp, tn, fn, fp]
error_colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
axes[1, 2].pie(error_vals, labels=error_labels, autopct='%1.1f%%',
               colors=error_colors, explode=(0.08, 0.05, 0.12, 0.05), shadow=True)
axes[1, 2].set_title('Prediction Distribution', fontsize=14, fontweight='bold')

plt.tight_layout(pad=3.0)
plt.savefig('milestone4_final_results.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# 16. Save results
results = {
    'approach': 'Focal Loss + Unsafe API Tags + Optimal Threshold',
    'unsafe_api_patterns': len(unsafe_functions),
    'training_samples': {'safe': SAFE_COUNT, 'vulnerable': VULN_COUNT, 'total': len(train_data)},
    'test_samples': {'safe': test_safe_count, 'vulnerable': test_vuln_count, 'total': len(test_data)},
    'optimal_threshold': float(best_threshold),
    'metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'auc_roc': float(auc_roc),
        'mcc': float(mcc),
        'average_precision': float(avg_precision)
    },
    'confusion_matrix': {'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)},
    'per_class': {
        'safe': {'precision': float(precision_class[0]), 'recall': float(recall_class[0]), 'f1': float(f1_class[0])},
        'vulnerable': {'precision': float(precision_class[1]), 'recall': float(recall_class[1]), 'f1': float(f1_class[1])}
    }
}

with open('milestone4_final_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# 17. final summary

print(f"""
╔══════════════════════════════════════════════════════════════╗
║         VULNERABILITY DETECTION - FINAL RESULTS               ║
╠══════════════════════════════════════════════════════════════╣
║                                                               ║
║  🔧 APPROACH:                                                 ║
║    • Focal Loss (alpha=0.25, gamma=2.0)                      ║
║    • {len(unsafe_functions)} Unsafe API Patterns for Feature Enhancement        ║
║    • Optimal Threshold: {best_threshold:.3f}                                  ║
║    • 15 Epochs + Cosine Restart Scheduler                    ║
║                                                               ║
║  DATA:                                                     ║
║    • Training: {len(train_data)} ({SAFE_COUNT}S + {VULN_COUNT}V)                              ║
║    • Test: {len(test_data)} ({test_safe_count}S + {test_vuln_count}V)                          ║
║                                                               ║
║  6 KEY METRICS:                                           ║
║    • Accuracy:  {accuracy:.4f} ({accuracy*100:.1f}%)                               ║
║    • Precision: {precision:.4f} ({precision*100:.1f}%)                               ║
║    • Recall:    {recall:.4f} ({recall*100:.1f}%)                               ║
║    • F1-Score:  {f1:.4f} ({f1*100:.1f}%)                               ║
║    • AUC-ROC:   {auc_roc:.4f} ({auc_roc*100:.1f}%)                               ║
║    • MCC:       {mcc:.4f} ({mcc*100:.1f}%)                               ║
║                                                               ║
║   CONFUSION: TP={tp} | FP={fp} | TN={tn} | FN={fn}                             ║
║   Vuln Found: {tp}/{test_vuln_count} ({(tp/test_vuln_count*100) if test_vuln_count > 0 else 0:.1f}%)                              ║
║                                                               ║
╚══════════════════════════════════════════════════════════════╝
""")


print("image and results: milestone4_final_results.png, milestone4_final_results.json")

Using Random test samples

In [ ]:
# Import all required libraries
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef
)
import re
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json
import random
import os
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# 1. Load Datasets
print("Load Datasets")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Load full datasets
with open("/content/PrimeVul-main/primevul_train .jsonl", "r") as f:
    train_full = [json.loads(line) for line in f]

with open("/content/PrimeVul-main/primevul_test .jsonl", "r") as f:
    test_full = [json.loads(line) for line in f]

print(f"Full training set: {len(train_full)} samples")
print(f"Full test set: {len(test_full)} samples")

# 2. Unsafe API tag Functions
print("Defining unsafe api")

# List of unsafe C/C++ functions
unsafe_functions = [
    "strcpy", "gets", "strcat", "sprintf", "scanf", "memcpy",
    "wcscpy", "_tcscpy", "lstrcpy", "wcscat", "_mbscpy",
    "vsprintf", "fscanf", "sscanf", "getwd", "recv", "read",
    "strlen", "malloc", "free", "realloc", "calloc"
]

def add_unsafe_api_tags(code: str):
    """
    Add unsafe API tags to code to help model identify vulnerable patterns
    """
    code_lower = code.lower()
    found = []

    for func in unsafe_functions:
        pattern = r"\b" + re.escape(func) + r"\s*\("
        if re.search(pattern, code_lower):
            found.append(func)

    # Add structured hint at end of code
    if found:
        code = code + "\n\n<unsafe_apis> " + " ".join(found) + " </unsafe_apis>"
    else:
        code = code + "\n\n<unsafe_apis> none </unsafe_apis>"

    return code

print(f" Defined {len(unsafe_functions)} unsafe API patterns")
print(f"Example unsafe functions: {unsafe_functions[:6]}...")

# Test the function
test_code = """
int vulnerable(char *input) {
    char buffer[10];
    strcpy(buffer, input);
    return 0;
}
"""
tagged_code = add_unsafe_api_tags(test_code)
print(f"\n Example tagged code:")
print(tagged_code[-100:])

# 3. Create balanced dataset (2000 Safe + 2000 Vulerable)
print("Preparing training data(2000 Safe + 2000 Vulnerable)")

# Separate by class
train_safe_all = [x for x in train_full if x['target'] == 0]
train_vuln_all = [x for x in train_full if x['target'] == 1]

print(f"Available safe samples: {len(train_safe_all)}")
print(f"Available vulnerable samples: {len(train_vuln_all)}")

# Shuffle and sample
random.shuffle(train_safe_all)
random.shuffle(train_vuln_all)

# Take 2000 from each class
train_safe_sampled = train_safe_all[:2000]
train_vuln_sampled = train_vuln_all[:2000]

# Combine and shuffle
train_data = train_safe_sampled + train_vuln_sampled
random.shuffle(train_data)

print(f"\nTraining Set (Balanced):")
print(f"  Safe (0): {len(train_safe_sampled)}")
print(f"  Vulnerable (1): {len(train_vuln_sampled)}")
print(f"  Total: {len(train_data)}")

# 4. Create random test data(1000 SAMPLES)
print("Preapring test data(Random 1000 Samples)")

# Shuffle and take random 1000 samples
random.shuffle(test_full)
test_data = test_full[:1000]

# Count class distribution
test_safe_count = sum(1 for x in test_data if x['target'] == 0)
test_vuln_count = sum(1 for x in test_data if x['target'] == 1)

print(f"Test Set (Random 1000):")
print(f"  Safe (0): {test_safe_count} ({test_safe_count/10:.1f}%)")
print(f"  Vulnerable (1): {test_vuln_count} ({test_vuln_count/10:.1f}%)")

# 5. Load tokenizer
print("loading tokenizer")

tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
print("Tokenizer loaded")

# 6. Train model with unsafe api tags
print("Training model with unsafe api tags")

# Initialize model
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/codebert-base",
    num_labels=2,
    ignore_mismatched_sizes=True
)
model.to(device)

# Prepare training data with unsafe API tags
print("Adding unsafe API tags to training data...")
train_codes = [add_unsafe_api_tags(x['func']) for x in train_data]
train_labels = [x['target'] for x in train_data]

# Tokenize training data
print("Tokenizing training data...")
train_encodings = tokenizer(
    train_codes,
    truncation=True,
    padding=True,
    max_length=512
)

# Create HuggingFace dataset
from datasets import Dataset as HFDataset
train_dataset = HFDataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels
})

# Training arguments
training_args = TrainingArguments(
    output_dir="./model_with_unsafe_tags",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=200,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    fp16=True if device.type == "cuda" else False,
    gradient_accumulation_steps=2,
)

# Train model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

print(f"\nTraining on {len(train_data)} samples with UNSAFE API TAGS...")
trainer.train()
print("Training complete!")

# Save model
trainer.save_model("./model_with_unsafe_tags")
tokenizer.save_pretrained("./model_with_unsafe_tags")
print(" Model saved to ./model_with_unsafe_tags")

model.eval()

# 7. prepare test data with unsafe api tags
print("prepare test data with unsafe api tags")
# Add unsafe API tags to test data
print("Adding unsafe API tags to test data...")
test_codes = [add_unsafe_api_tags(item['func']) for item in test_data]
test_labels = [item['target'] for item in test_data]

print(f"Test samples: {len(test_data)}")
print(f"  Safe (0): {test_safe_count}")
print(f"  Vulnerable (1): {test_vuln_count}")

# 8. create dataset and data loader
class TestDataset(Dataset):
    def __init__(self, codes, labels, tokenizer, max_length=512):
        self.codes = codes
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.codes[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx])
        }

test_dataset = TestDataset(test_codes, test_labels, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Test batches: {len(test_loader)}")

# 9. Generate Predictions
print("Generating Predictions")
all_predictions = []
all_probabilities = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probabilities = F.softmax(outputs.logits, dim=1)
        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(probabilities[:, 1].cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)
all_labels = np.array(all_labels)

print(f"Predictions generated for {len(all_labels)} samples")

# 10. calculate 6 key metrics
print(" 6 Key performance metrics(with unsafe api tags)")

# Confusion matrix elements
tn, fp, fn, tp = confusion_matrix(all_labels, all_predictions).ravel()

# Calculate 6 key metrics
accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions, zero_division=0)
recall = recall_score(all_labels, all_predictions, zero_division=0)
f1 = f1_score(all_labels, all_predictions, zero_division=0)
auc_roc = roc_auc_score(all_labels, all_probabilities)
mcc = matthews_corrcoef(all_labels, all_predictions)

# 11. Display 6 key results
print(f"""
╔══════════════════════════════════════════════════════════════╗
║             6 key metrics with unsafe api tags          ║
╠══════════════════════════════════════════════════════════════╣
║                                                               ║
║    Key metrics:                                              ║
║    • Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)                              ║
║    • Precision: {precision:.4f} ({precision*100:.2f}%)                              ║
║    • Recall:    {recall:.4f} ({recall*100:.2f}%)                              ║
║    • F1-Score:  {f1:.4f} ({f1*100:.2f}%)                              ║
║    • AUC-ROC:   {auc_roc:.4f} ({auc_roc*100:.2f}%)                              ║
║    • MCC:       {mcc:.4f} ({mcc*100:.2f}%)                              ║
║                                                               ║
╠══════════════════════════════════════════════════════════════╣
║    Confusion matrix:                                         ║
║    • True Positives:  {tp:<6} (Correctly detected vulns)     ║
║    • True Negatives:  {tn:<6} (Correctly identified safe)    ║
║    • False Positives: {fp:<6} (False alarms)                 ║
║    • False Negatives: {fn:<6} (Missed vulnerabilities)       ║
║                                                               ║
╠══════════════════════════════════════════════════════════════╣
║    Dataset information:                                      ║
║    • Training: 2000 Safe + 2000 Vulnerable (Balanced)       ║
║    • Testing:  {test_safe_count} Safe + {test_vuln_count} Vulnerable (Random 1000)         ║
║    • Unsafe API Tags: ENABLED                                ║
║                                                               ║
╚══════════════════════════════════════════════════════════════╝
""")

# 12. Metrics table format
print("Metrics summary table")

print(f"""
┌────────────────────────────┬──────────┬───────────────┐
│ METRIC                     │ VALUE    │ INTERPRETATION│
├────────────────────────────┼──────────┼───────────────┤
│ 1. Accuracy                │ {accuracy:.4f}  │ {accuracy*100:.1f}% correct      │
│ 2. Precision               │ {precision:.4f}  │ {precision*100:.1f}% true positives│
│ 3. Recall (Sensitivity)    │ {recall:.4f}  │ {recall*100:.1f}% vulns found   │
│ 4. F1-Score                │ {f1:.4f}  │ {f1*100:.1f}% balanced      │
│ 5. AUC-ROC                 │ {auc_roc:.4f}  │ {auc_roc*100:.1f}% discrimination│
│ 6. MCC                     │ {mcc:.4f}  │ {mcc*100:.1f}% correlation   │
└────────────────────────────┴──────────┴───────────────┘
""")

# 13. SAVE RESULTS
results = {
    'approach': 'WITH Unsafe API Tags',
    'training': '2000 Safe + 2000 Vulnerable (Balanced)',
    'testing': f'{test_safe_count} Safe + {test_vuln_count} Vulnerable (Random 1000)',
    'metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'auc_roc': float(auc_roc),
        'mcc': float(mcc)
    },
    'confusion_matrix': {
        'true_positives': int(tp),
        'true_negatives': int(tn),
        'false_positives': int(fp),
        'false_negatives': int(fn)
    }
}

with open('milestone4_results_with_tags.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n Results are saved to 'milestone4_results_with_tags.json")


Without Unsafe API (80-20 Safe-Vulnerable samples)

In [ ]:

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    matthews_corrcoef
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json
import random
import warnings
warnings.filterwarnings('ignore')

# 1. Load data
print("\n loading datasets")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

with open("/content/PrimeVul-main/primevul_train .jsonl", "r") as f:
    train_full = [json.loads(line) for line in f]

with open("/content/PrimeVul-main/primevul_test .jsonl", "r") as f:
    test_full = [json.loads(line) for line in f]

# 2. Training : 2000 safe + 2000 vulnerable
train_safe_all = [x for x in train_full if x['target'] == 0]
train_vuln_all = [x for x in train_full if x['target'] == 1]

random.shuffle(train_safe_all)
random.shuffle(train_vuln_all)

VULN_COUNT = min(len(train_vuln_all), 2000)
SAFE_COUNT = VULN_COUNT

train_data = train_safe_all[:SAFE_COUNT] + train_vuln_all[:VULN_COUNT]
random.shuffle(train_data)

print(f"Training: {SAFE_COUNT} Safe + {VULN_COUNT} Vuln = {len(train_data)}")

# 3. Test : 800 safe + 200 vulnerable
test_safe_all = [x for x in test_full if x['target'] == 0]
test_vuln_all = [x for x in test_full if x['target'] == 1]

random.shuffle(test_safe_all)
random.shuffle(test_vuln_all)

TEST_SAFE = 800
TEST_VULN = min(len(test_vuln_all), 200)

test_data = test_safe_all[:TEST_SAFE] + test_vuln_all[:TEST_VULN]
random.shuffle(test_data)

test_safe_count = TEST_SAFE
test_vuln_count = TEST_VULN

print(f"Test: {test_safe_count} Safe + {test_vuln_count} Vuln = {len(test_data)}")

# 4. Tokenize
print("\n Tokenizing")
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

train_codes = [x['func'] for x in train_data]
train_labels = [x['target'] for x in train_data]
test_codes = [item['func'] for item in test_data]
test_labels = [item['target'] for item in test_data]

train_encodings = tokenizer(train_codes, truncation=True, padding=True, max_length=256, return_tensors='pt')
test_encodings = tokenizer(test_codes, truncation=True, padding=True, max_length=256, return_tensors='pt')
print(f"Train: {train_encodings['input_ids'].shape}, Test: {test_encodings['input_ids'].shape}")

# 5. Dataset
from datasets import Dataset as HFDataset
train_dataset = HFDataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels
})


# 6. Focal loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

# 7. Model
print("\n Loading model")
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/codebert-base", num_labels=2, ignore_mismatched_sizes=True
)
model.to(device)

focal_loss_fn = FocalLoss(alpha=0.25, gamma=2.0)

class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = focal_loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# 8. Train
print("\n Training with Focal Loss")

training_args = TrainingArguments(
    output_dir="./focal_model_final",
    num_train_epochs=15,
    per_device_train_batch_size=16,
    learning_rate=5e-5,
    weight_decay=0.02,
    warmup_ratio=0.1,
    logging_steps=20,
    save_strategy="no",
    report_to="none",
    fp16=True if device.type == "cuda" else False,
    gradient_accumulation_steps=2,
    lr_scheduler_type="cosine_with_restarts",
    dataloader_pin_memory=False,
)

trainer = FocalLossTrainer(model=model, args=training_args, train_dataset=train_dataset)
trainer.train()
model.eval()

# 9. Predictions

print("\n Generating predictions...")

test_dataset = TensorDataset(
    test_encodings['input_ids'],
    test_encodings['attention_mask'],
    torch.tensor(test_labels, dtype=torch.long)
)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

all_probabilities = []
all_labels_list = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=1)
        all_probabilities.extend(probs[:, 1].cpu().numpy())
        all_labels_list.extend(labels.numpy())

all_probabilities = np.array(all_probabilities)
all_labels = np.array(all_labels_list)

# 10. Optimal threshold
print("\n Finding optimal threshold...")

best_f1 = 0
best_threshold = 0.5
best_predictions = None
best_precision = 0
best_recall = 0

for threshold in np.arange(0.01, 0.99, 0.005):
    predictions = (all_probabilities >= threshold).astype(int)
    if predictions.sum() < 5:
        continue

    prec = precision_score(all_labels, predictions, zero_division=0)
    rec = recall_score(all_labels, predictions, zero_division=0)
    f1_val = f1_score(all_labels, predictions, zero_division=0)

    if f1_val > best_f1:
        best_f1 = f1_val
        best_threshold = threshold
        best_predictions = predictions.copy()
        best_precision = prec
        best_recall = rec

print(f"Optimal threshold: {best_threshold:.3f}")
print(f"Best F1-Score: {best_f1:.4f}")

# 11. Final metrics
all_predictions = best_predictions

tn, fp, fn, tp = confusion_matrix(all_labels, all_predictions).ravel()

accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions, zero_division=0)
recall = recall_score(all_labels, all_predictions, zero_division=0)
f1 = f1_score(all_labels, all_predictions, zero_division=0)
auc_roc = roc_auc_score(all_labels, all_probabilities)
avg_precision = average_precision_score(all_labels, all_probabilities)
mcc = matthews_corrcoef(all_labels, all_predictions)

precision_class, recall_class, f1_class, _ = precision_recall_fscore_support(
    all_labels, all_predictions, zero_division=0
)


# 12. Display results
print("FINAL RESULTS")

print(f"""
╔══════════════════════════════════════╦════════════╗
║ METRIC                               ║ VALUE      ║
╠══════════════════════════════════════╬════════════╣
║ 1. Accuracy                          ║ {accuracy:.4f}    ║
║ 2. Precision (Vulnerable)            ║ {precision:.4f}    ║
║ 3. Recall / Sensitivity              ║ {recall:.4f}    ║
║ 4. F1-Score (Vulnerable)             ║ {f1:.4f}    ║
║ 5. AUC-ROC                           ║ {auc_roc:.4f}    ║
║ 6. MCC (Matthews Corr. Coeff.)       ║ {mcc:.4f}    ║
╚══════════════════════════════════════╩════════════╝
""")

print(f"""
 CONFUSION MATRIX:
┌────────────────────┬──────────────┬──────────────────┐
│                    │ Pred. SAFE   │ Pred. VULNERABLE │
├────────────────────┼──────────────┼──────────────────┤
│ Actually SAFE      │     {tn:<8} │       {fp:<10} │
│ Actually VULNERABLE│     {fn:<8} │       {tp:<10} │
└────────────────────┴──────────────┴──────────────────┘

 Vulnerabilities Found: {tp}/{test_vuln_count} ({(tp/test_vuln_count*100) if test_vuln_count > 0 else 0:.1f}%)
 Safe Code Correct: {tn}/{test_safe_count} ({(tn/test_safe_count*100) if test_safe_count > 0 else 0:.1f}%)
 False Alarms: {fp}
 Missed Vulnerabilities: {fn}
""")

print(f"""
📋 CLASSIFICATION REPORT:
{classification_report(all_labels, all_predictions, target_names=['Safe (0)', 'Vulnerable (1)'], zero_division=0)}
""")

# 13. Visualization
print("Generating visualizations")

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
fig.suptitle(f'Vulnerability Detection Results (Focal Loss)\nAcc={accuracy:.4f} | Prec={precision:.4f} | Rec={recall:.4f} | F1={f1:.4f} | AUC={auc_roc:.4f} | MCC={mcc:.4f}',
             fontsize=11, fontweight='bold', y=0.98)

# 1. Confusion Matrix
sns.heatmap([[tn, fp], [fn, tp]], annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred SAFE', 'Pred VULNERABLE'],
            yticklabels=['True SAFE', 'True VULNERABLE'],
            ax=axes[0, 0], cbar_kws={'label': 'Count'},
            annot_kws={'size': 16, 'fontweight': 'bold'})
axes[0, 0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')

# 2. ROC Curve
fpr_curve, tpr_curve, _ = roc_curve(all_labels, all_probabilities)
axes[0, 1].plot(fpr_curve, tpr_curve, 'b-', linewidth=3, label=f'AUC = {auc_roc:.4f}')
axes[0, 1].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random')
axes[0, 1].fill_between(fpr_curve, tpr_curve, alpha=0.3, color='blue')
axes[0, 1].set_xlabel('False Positive Rate', fontsize=12)
axes[0, 1].set_ylabel('True Positive Rate', fontsize=12)
axes[0, 1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[0, 1].legend(loc='lower right', fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

# 3. Precision-Recall Curve
prec_curve, rec_curve, _ = precision_recall_curve(all_labels, all_probabilities)
axes[0, 2].plot(rec_curve, prec_curve, 'g-', linewidth=3, label=f'AP = {avg_precision:.4f}')
axes[0, 2].set_xlabel('Recall', fontsize=12)
axes[0, 2].set_ylabel('Precision', fontsize=12)
axes[0, 2].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[0, 2].legend(loc='upper right', fontsize=11)
axes[0, 2].grid(True, alpha=0.3)

# 4. 6 Key Metrics
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'MCC']
metric_vals = [accuracy, precision, recall, f1, auc_roc, mcc]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#1abc9c', '#9b59b6']
bars = axes[1, 0].bar(metric_names, metric_vals, color=colors, edgecolor='black', linewidth=2)
axes[1, 0].set_ylim([0, 1.15])
axes[1, 0].set_title('6 Key Performance Metrics', fontsize=14, fontweight='bold')
for bar, val in zip(bars, metric_vals):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
                    f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 5. Per-Class Performance
class_names = ['Safe (0)', 'Vulnerable (1)']
f1_vals = [f1_class[0], f1_class[1]]
prec_vals = [precision_class[0], precision_class[1]]
rec_vals = [recall_class[0], recall_class[1]]

x = np.arange(len(class_names))
width = 0.25
axes[1, 1].bar(x - width, prec_vals, width, label='Precision', color='#3498db', edgecolor='black')
axes[1, 1].bar(x, rec_vals, width, label='Recall', color='#e74c3c', edgecolor='black')
axes[1, 1].bar(x + width, f1_vals, width, label='F1-Score', color='#2ecc71', edgecolor='black')
axes[1, 1].set_xlabel('Class', fontsize=12)
axes[1, 1].set_title('Per-Class Performance', fontsize=14, fontweight='bold')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(class_names, fontsize=11)
axes[1, 1].legend(fontsize=11)
axes[1, 1].set_ylim([0, 1.1])
axes[1, 1].grid(True, alpha=0.3, axis='y')

# 6. Pie Chart
error_labels = ['True Positive\n(Vuln Detected)', 'True Negative\n(Safe Detected)',
                'False Negative\n(Vuln Missed)', 'False Positive\n(False Alarm)']
error_vals = [tp, tn, fn, fp]
error_colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
axes[1, 2].pie(error_vals, labels=error_labels, autopct='%1.1f%%',
               colors=error_colors, explode=(0.08, 0.05, 0.12, 0.05), shadow=True)
axes[1, 2].set_title('Prediction Distribution', fontsize=14, fontweight='bold')

plt.tight_layout(pad=3.0)
plt.savefig('milestone4_final_results.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()


# 14. Save results
results = {
    'approach': 'Focal Loss + Optimal Threshold',
    'training_samples': {'safe': SAFE_COUNT, 'vulnerable': VULN_COUNT, 'total': len(train_data)},
    'test_samples': {'safe': test_safe_count, 'vulnerable': test_vuln_count, 'total': len(test_data)},
    'optimal_threshold': float(best_threshold),
    'metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'auc_roc': float(auc_roc),
        'mcc': float(mcc),
        'average_precision': float(avg_precision)
    },
    'confusion_matrix': {'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)},
    'per_class': {
        'safe': {'precision': float(precision_class[0]), 'recall': float(recall_class[0]), 'f1': float(f1_class[0])},
        'vulnerable': {'precision': float(precision_class[1]), 'recall': float(recall_class[1]), 'f1': float(f1_class[1])}
    }
}

with open('milestone4_final_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# 15. Final summary

print(f"""
╔══════════════════════════════════════════════════════════════╗
║         VULNERABILITY DETECTION - FINAL RESULTS               ║
╠══════════════════════════════════════════════════════════════╣
║                                                               ║
║  🔧 APPROACH:                                                 ║
║    • Focal Loss (alpha=0.25, gamma=2.0)                      ║
║    • Optimal Threshold: {best_threshold:.3f}                                  ║
║    • 15 Epochs + Cosine Restart Scheduler                    ║
║                                                               ║
║  📊 DATA:                                                     ║
║    • Training: {len(train_data)} ({SAFE_COUNT}S + {VULN_COUNT}V)                              ║
║    • Test: {len(test_data)} ({test_safe_count}S + {test_vuln_count}V)                          ║
║                                                               ║
║  🎯 6 KEY METRICS:                                           ║
║    • Accuracy:  {accuracy:.4f} ({accuracy*100:.1f}%)                               ║
║    • Precision: {precision:.4f} ({precision*100:.1f}%)                               ║
║    • Recall:    {recall:.4f} ({recall*100:.1f}%)                               ║
║    • F1-Score:  {f1:.4f} ({f1*100:.1f}%)                               ║
║    • AUC-ROC:   {auc_roc:.4f} ({auc_roc*100:.1f}%)                               ║
║    • MCC:       {mcc:.4f} ({mcc*100:.1f}%)                               ║
║                                                               ║
║  📊 CONFUSION: TP={tp} | FP={fp} | TN={tn} | FN={fn}                             ║
║  📊 Vuln Found: {tp}/{test_vuln_count} ({(tp/test_vuln_count*100) if test_vuln_count > 0 else 0:.1f}%)                              ║
║                                                               ║
╚══════════════════════════════════════════════════════════════╝
""")


print(" image and results file generated: milestone4_final_results.png, milestone4_final_results.json")